# Hotel Booking Analysis — Mac-compatible corrected edition
This copy preserves the original notebook's analysis while fixing non-portable paths, a percentile-loop error, deprecated plotting calls, and missing local image references. Run with `hotel_bookings.csv` in the same directory. Original tutorial notebook remains unchanged.

## 1.. Lets read data !

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
csv_path = Path("hotel_bookings.csv")
if not csv_path.exists():
    csv_path = Path.cwd().parent / "hotel_bookings.csv"
df = pd.read_csv(csv_path)

In [ ]:
type(df)

In [ ]:
df.head(4)

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
'''

Categorical data refers to a data type that can be stored 
into groups/categories/labels ..

Examples of categorical variables are :  
age group, educational level,blood type etc.. 


Numerical data refers to the data that is in the form of numbers, 
Examples of numerical data are height, weight, age etc.. 

Numerical data has two categories: discrete data and continuous data


Discrete data : It basically takes countable numbers like 1, 2, 3, 4, 
                5, and so on. 
                In case of infinity, these numbers will keep going on...
                age of a fly : 8 , 9 day etc..
                
Continuous data : which is continuous in nature 
                  amount of sugar , 11.2 kg  , temp of a city  , 
                  your bank balance !
                  
For example, salary levels and performance classifications are 
discrete variables,  whereas height and weight are continuous variables.


'''

In [ ]:
'''

Categorical data has : Object & bool data-types 
Numerical data have : Integer & Float data-type

'''

In [ ]:
'''

Variations of int are : ('int64','int32','int16') in numpy library..

Int16 is a 16 bit signed integer , it means it can store both positive & negative values
int16 has has a range of  (2^15 − 1) to -2^15 
int16 has a length of 16 bits (2 bytes).. ie Int16 uses 16 bits 


Int32 is a 32 bit signed integer , it means it stores both positive & negative values
int32 has has a range of (2³¹ − 1) to  -2^31
int32 has a length of 32 bits (4 bytes),, ie Int32 uses 32 bits


Int64 is a 64 bit signed integer , it means it can store both positive & negative values
int64 has has a range of  (2^63 − 1) to -2^63 
int64 has a length of 64 bits (8 bytes) , ie Int64 uses 64 bits.
             

The only difference is that int64 has max range of storing numbers , then comes int32 , then 16 , then int8

That means that Int64’s take up twice as much memory-and doing operations on them may be a lot slower in some machine architectures.

However, Int64’s can represent numbers much more accurately than 32 bit floats.They also allow much larger numbers to be stored..

'''

## 2.. Lets do Data cleaning !
    Removing invalid rows

In [ ]:
df.head(3)

In [ ]:
df.columns

In [ ]:
filter1 = (df['children']==0) & (df['adults']==0) & (df['babies']==0)

## Adults,babies & children cant be zero at a same time bcz booking couldn't be possible if these 3 attributes are 0 ..

In [ ]:
filter1

In [ ]:
df[filter1]

In [ ]:
df.shape

In [ ]:
df[filter1].shape

In [ ]:
df[~filter1].shape

In [ ]:
df2 = df[~filter1]

In [ ]:
len(df2[(df2['children']==1) & (df2['adults']==0) & (df2['babies']==0)])

### Removing Duplicated data

In [ ]:
df2.duplicated()

In [ ]:
type(df2.duplicated())

In [ ]:
df2.duplicated().sum()

In [ ]:
data= df2.drop_duplicates()

In [ ]:
data.shape

## 3.. Performing descriptive analysis !

In [ ]:
data.columns

In [ ]:
data[['lead_time' , 'total_of_special_requests' , 'adr']].describe().T

## getting (mean, median , std , percentile) of above features

In [ ]:
data.info()

In [ ]:
'''

12 features belong to object data-type , ie.. in context to Python ,
they belong to string data-type

16 features belong to int64 nature 

4 features belong to float64 nature  ,


The memory usage of a DataFrame (including the index) is shown when calling the info(). 

 
 
The + symbol indicates that the true memory usage could be higher,  
because pandas does not count the memory used by values in columns 
with dtype=object


Passing memory_usage='deep' will enable a more accurate 
memory usage report ..


'''

In [ ]:
data.info(memory_usage='deep')

In [ ]:
'''
print quantile values ..

'''

In [ ]:
for col in ['lead_time', 'total_of_special_requests', 'adr']:
    print('Feature:', col)
    for percentile in range(90, 101):
        quantile_value = data[col].quantile(percentile / 100)
        print(f'{percentile}th percentile: {quantile_value:.2f}')
    print()

In [ ]:
np.quantile(data['lead_time'] , q=0.9)

In [ ]:
'''

"adr" feature seems to have Outlier as 99th percentile value 
is 261 but 100th percentile(max value) is 5400 .. 


'''

## 4.. Perform Spatial Analysis
        Where do the guests come from ?

In [ ]:
not_cancelled = data[data['is_canceled']==0]

In [ ]:
not_cancelled.head(3)

In [ ]:
country_wise_data = not_cancelled['country'].value_counts().reset_index()

In [ ]:
country_wise_data.columns = ['country' , 'No of guests']

In [ ]:
country_wise_data

In [ ]:
# Install once in your Jupyter environment if needed:
# %pip install plotly chart-studio

In [ ]:
import plotly.graph_objs as go
import plotly.express as px
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

In [ ]:
# show on map

map_guest = px.choropleth(data_frame = country_wise_data , 
              locations= country_wise_data['country'] , 
              color=country_wise_data['No of guests'] , 
              hover_name=country_wise_data['country'] , 
              title= "Home country of Guests"
          
             )

In [ ]:
map_guest.show()

#### Most guests are from Portugal and other countries in Europe

## 5.. Is any difference between assigned and reserved room types or not ?

In [ ]:
data.columns

In [ ]:
pivot = pd.crosstab(index = data['reserved_room_type'] , columns=data['assigned_room_type'] , margins=True  )

In [ ]:
pivot

In [ ]:
'''

## lets find meaningful insight from this :

for A category room , 56436 folks have reserved "A" & 45850 folks get assigned_room as "A".. & rest are unable to get !

for B category room , 996 folks have reserved "B" &  872 folks get assigned_room as "B".. & rest are unable to get !


'''

In [ ]:
### Lets normalize above stuff to get more meaningful insights !

In [ ]:
## we will say just normalize over row , hence we need to pass normalize = 'index'

pivot_normalize = pd.crosstab(index = data['reserved_room_type'] , columns=data['assigned_room_type'] , margins=True ,normalize='index' ).round(2)*100

In [ ]:
pivot_normalize

In [ ]:
'''

Q.. Is any difference between assigned and reserved room type ?
Ans : Yes 


'''

##  6.. Bookings by market segment

In [ ]:
data['market_segment'].value_counts()

In [ ]:
data['market_segment'].value_counts().values

In [ ]:
data['market_segment'].value_counts().index

In [ ]:
# pie plot

fig = px.pie(data , 
      values = data['market_segment'].value_counts().values , 
      names = data['market_segment'].value_counts().index)

In [ ]:
fig.show()

In [ ]:
'''
Most of the bookings have been done in Online mode 

'''

### 6b) Analysing Avg.price per night (ADR) of various room-types for all the market segment ..

In [ ]:
data.columns

In [ ]:
plt.figure(figsize=(20, 10))
sns.barplot(x="market_segment", y="adr", hue="reserved_room_type", data=data, errorbar=None)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7.. Total guests arrival on each day : 
         Is there any pattern in guests arrival , ie whether 
         Guests number have increased or not ? 

In [ ]:
data.head(3)

In [ ]:
data['arrival_date_month'].unique()

In [ ]:
dict_month = {'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12,
       'January':1, 'February':2, 'March':3, 'April':4, 'May':5, 'June':6}

In [ ]:
import warnings
from warnings import filterwarnings
filterwarnings('ignore')

In [ ]:
data['arrival_date_month_index'] = data['arrival_date_month'].map(dict_month)

In [ ]:
data.columns

In [ ]:
data[['arrival_date_year',
       'arrival_date_month_index' , 'arrival_date_day_of_month']]

In [ ]:
data['arrival_date'] = data['arrival_date_year'].astype(str) + '-' + data['arrival_date_month_index'].astype(str) + '-' + data['arrival_date_day_of_month'].astype(str)


'''
we need to use .astype(str) to convert int values to string , 
otherwise we are unable to perform this string concatenation operation ..
'''

In [ ]:
data.head(2)

In [ ]:
data.columns

In [ ]:
data[['adults', 'children', 'babies']]

In [ ]:
data['Total_guests'] = data['adults'] + data['children'] + data['babies']

In [ ]:
data.head(2)

In [ ]:
data.columns

In [ ]:
data[['arrival_date', 'Total_guests']]

In [ ]:
dataNoCancel = data[data['is_canceled']==0]

In [ ]:
guest_arrival_series = dataNoCancel.groupby(['arrival_date'])['Total_guests'].sum()

In [ ]:
guest_arrival_series.plot(figsize=(10,6))

In [ ]:
'''
Q.. Is there any pattern ?
Ans .. No , there is no visible pattern in guests arrival 
in this line-plot as we have some un-even trend ..

'''

## 8.. Analysing distribution of "guests arrival"
     lets plot distribution of "guests arrival"

In [ ]:
guest_arrival_series

In [ ]:
type(guest_arrival_series)

### this is a series having indexes as "arrival_dates" ..

In [ ]:
guest_arrival_series.values

## lets obtain array representation of Series so that it is easy to get our distribution ..

In [ ]:
sns.histplot(guest_arrival_series.values, kde=True)
plt.xlabel('Daily guest arrivals')
plt.show()

In [ ]:
sns.displot(guest_arrival_series.values )

In [ ]:
sns.displot(guest_arrival_series.values , kind='kde' )

In [ ]:
'''


we can achieve distribution plot by smoothening our histogram using KDE  ie PDF is a smoothen form of your histogram !
ie histogram -->> apply KDE -->> we will get distribution plot 



This is called density plot bcz here height representshow many pts exists at each of these intervals 
or how dense each of the region is !



density(distribution) plot : at a point , what is a density of a data pt. ?
                or
how many percentage of data pts available at some particular pt .. ?
                or 
What is the % of data points that I will encounter at any point ? 
                or
what is the probability of certain data pt in whole data ?
                or 
prob that my data has value has some specific value 



Note : Above distribution is close to Gaussian/normal distribution , 


'''

In [ ]:
# Original tutorial referenced a local Windows image that is not included in this repository.
# Refer to the explanatory notes in the following cell instead.

In [ ]:
'''

If mean & median are equal , it means 
distribution is symmetrical & bell-shaped 

If mean > median , it means distribution is not symmetrical 
& it is Right skewed (positively skewed )

If mean < median , it means distribution is not symmetrical 
& it is Right skewed (negatively skewed )


'''

#### Properties of Normal/Gaussian Distribution : 
     Bell shaped curve 
     Symmetrical distribution
     68-95-99 rule

In [ ]:
np.mean(guest_arrival_series.values)

In [ ]:
np.median(guest_arrival_series.values)

## mean & median are approx equals , 
## it means distribution is a symmetrical & bell-shaped 

In [ ]:
# Original tutorial referenced a local Windows image that is not included in this repository.
# The mean, median and standard deviation are calculated in adjacent cells.

In [ ]:
np.std(guest_arrival_series.values)

In [ ]:
'''

between 1 std dev ie between u-sigma to u+sigma ie between 100 to 213 , 
we have approx 68% of data pts 
ie , approx 68% of total guests arrival values lies between interval of 100 to 213


between 2 std dev ie between u-2sigma to u+2sigma ie between 46 to 269 , 
we have approx 95% of data pts 
ie , approx 95% of total guests arrival values lies between interval of 46 to 269


between 3 std dev ie between u-3sigma to u+3sigma ie between 0 to 320 , 
we have approx 99.4% of data pts 
ie , approx 99.7% of total guests arrival values lies between interval of 0 to 320


'''